In [ ]:
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import cutouts_to_patches.cutouts_to_patches as cutouts_to_patches

In [ ]:
from nemi import NEMI, SingleNemi

In [ ]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2_TESTING",
                                          run_id="itest_d7af6005",
                                          dataset_name="cutout_dataset_creation.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io"
                                          )

source.print_available_channels()

In [ ]:
# cutouts_dataloader = cutouts_dataset.get_cutout_loader(source=source, subset=True, subsample_per_chunk = 64, num_sample_chunks = 1, n_workers=1, batch_size=64)
data_channels = ['Eta','Salt','Theta','U','V','W','gradb2','oceTAUX','oceTAUY','gradrho2','turner_angle','strain_n','strain_s','strain_mag','divergence','relative_vorticity','coriolis_f']

cutouts_dataloader = cutouts_dataset.get_cutout_loader(data_channels=data_channels, source=source, subset=False, subsample_per_chunk = 64, num_sample_chunks = 1, n_workers=1, batch_size=64)

In [ ]:
patch_size = 8
patched_loader = cutouts_to_patches.PatchedDataLoader(cutouts_dataloader, patch_size=patch_size)
print(len(patched_loader))

patches = torch.cat(
    [patches.reshape(patches.shape[0] * patches.shape[1], -1) for patches in patched_loader], dim=0
).numpy()

print(patches.shape)

In [ ]:
# Run NEMI on the patches (GPU: cuML UMAP + clustering)
nemi = NEMI(params={
    "device": "gpu",
    "embedding_dict":  {"n_components": 3, "n_neighbors": 200, "min_dist": 0.0},
    "clustering_dict": {"method": "hdbscan", "min_cluster_size": 500, "min_samples": 500},
})

# single member to start; bump n (with assess_overlap=True) for the ensemble
nemi.run(patches, n=1, output="nemi_out.npz")

labels    = nemi.clusters      # (N,) cluster label per patch
embedding = nemi.embedding     # (N, 3) UMAP embedding
print("labels", labels.shape, "| embedding", embedding.shape)